Import libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

Fetch Adult dataset from sklearn

In [ ]:
dataset = fetch_openml(name='adult', version=2, as_frame=True)
df = dataset.frame

# Display the first few rows of the dataset
print(df.head())

# Binary encode the target variable
le = LabelEncoder()
df['class'] = le.fit_transform(df['class'])

# Display the first few rows of the dataset, to check if the target variable is now binary
print(df.head())

   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  class  
0       

Split data into training and test sets

In [ ]:
# Split the data into features and target
X = df.drop(columns='class')
y = df['class']

# Further split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Display the shape of the training set
X_train.shape, X_test.shape



((34189, 14), (14653, 14))

We will focus on the 'occupation' variable, which is a high cardinality variable

In [ ]:
# Display unique values in the 'occupation' column
X_train['occupation'].value_counts()


occupation
Prof-specialty       4333
Craft-repair         4282
Exec-managerial      4256
Adm-clerical         3918
Sales                3880
Other-service        3459
Machine-op-inspct    2137
Transport-moving     1640
Handlers-cleaners    1418
Farming-fishing      1030
Tech-support          954
Protective-serv       714
Priv-house-serv       180
Armed-Forces           11
Name: count, dtype: int64

**Supervised Ratio Encoding**

Supervised ratio encoding involves encoding a categorical variable based on the ratio of the target variable's mean for each category.

In [ ]:
# Function to calculate supervised ratio encoding
def supervised_ratio_encoding(X, y, column):
    temp = pd.concat([X[column].astype(str), y], axis=1)
    means = temp.groupby(column)['class'].mean()
    return means, X[column].map(means)

# Apply supervised ratio encoding to the training set
occupation_means, X_train['occupation_supervised_ratio'] = supervised_ratio_encoding(X_train, y_train, 'occupation')

# Apply the same encoding to the test set
X_test['occupation_supervised_ratio'] = X_test['occupation'].map(occupation_means)

# Display the first few rows with the new encoded feature
X_train.head()


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,occupation_supervised_ratio,occupation_woe
42392,25.0,Private,188767.0,HS-grad,9.0,Never-married,Craft-repair,Own-child,White,Male,0.0,0.0,40.0,United-States,0.226296,-0.135198
14623,64.0,Private,286732.0,HS-grad,9.0,Widowed,Sales,Not-in-family,White,Female,0.0,0.0,17.0,United-States,0.272680,0.113081
27411,29.0,Private,253801.0,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0.0,0.0,40.0,Ecuador,0.226296,-0.135198
1288,28.0,Private,334032.0,Assoc-voc,11.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0.0,0.0,50.0,United-States,0.226296,-0.135198
7078,22.0,Private,173004.0,HS-grad,9.0,Never-married,Machine-op-inspct,Other-relative,Black,Male,0.0,0.0,1.0,United-States,0.123538,-0.865201


**Weight of Evidence (WoE) encoding**

Weight of Evidence encoding transforms categorical variables based on the distribution of the target variable within each category.

OR you can use the predefined module *WoEEncoder* in the feature_engine library.



In [ ]:
# Function to calculate WoE
def woe_encoding(X, y, column):
    temp = pd.concat([X[column].astype(str), y], axis=1)
    dist = temp.groupby(column).agg({'class': ['count', 'sum']})
    dist.columns = ['count', 'sum']
    dist['non_event'] = dist['count'] - dist['sum']
    dist['event_rate'] = dist['sum'] / dist['sum'].sum()
    dist['non_event_rate'] = dist['non_event'] / dist['non_event'].sum()
    dist['woe'] = np.log(dist['event_rate'] / dist['non_event_rate'])
    return dist['woe'], X[column].map(dist['woe'])

# Apply WoE encoding to the training set
occupation_woe, X_train['occupation_woe'] = woe_encoding(X_train, y_train, 'occupation')

# Apply the same encoding to the test set
X_test['occupation_woe'] = X_test['occupation'].map(occupation_woe)

# Display the first few rows with the new encoded feature
X_train.head()


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,occupation_supervised_ratio,occupation_woe
42392,25.0,Private,188767.0,HS-grad,9.0,Never-married,Craft-repair,Own-child,White,Male,0.0,0.0,40.0,United-States,0.226296,-0.086524
14623,64.0,Private,286732.0,HS-grad,9.0,Widowed,Sales,Not-in-family,White,Female,0.0,0.0,17.0,United-States,0.272680,0.161755
27411,29.0,Private,253801.0,HS-grad,9.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0.0,0.0,40.0,Ecuador,0.226296,-0.086524
1288,28.0,Private,334032.0,Assoc-voc,11.0,Married-civ-spouse,Craft-repair,Husband,White,Male,0.0,0.0,50.0,United-States,0.226296,-0.086524
7078,22.0,Private,173004.0,HS-grad,9.0,Never-married,Machine-op-inspct,Other-relative,Black,Male,0.0,0.0,1.0,United-States,0.123538,-0.816527


Select the encoded features you want to include in your model before training the model. Select the same features when predicting scores on the test set using that model.